# VPS Setup

How to provision and harden a production VPS. This guide covers the full setup from a fresh server to a production-ready host running Coolify.

**Target:** Hostinger KVM2 (Ubuntu 22.04 LTS). Applies to any Ubuntu VPS.

## 1. First Access

Hostinger provides a root password via email. SSH in immediately and change it.

```bash
ssh root@YOUR_SERVER_IP
```

### Set hostname

```bash
hostnamectl set-hostname saucemachine
```

### Update packages

```bash
apt update && apt upgrade -y
apt install -y curl wget git ufw fail2ban unattended-upgrades
```

## 2. SSH Key Authentication

Password auth should be disabled. Use SSH keys only.

### On your local machine

```bash
# Generate a key if you do not have one
ssh-keygen -t ed25519 -C "you@example.com"

# Copy to server
ssh-copy-id -i ~/.ssh/id_ed25519.pub root@YOUR_SERVER_IP
```

### On the server: disable password auth

```bash
# Edit /etc/ssh/sshd_config
PasswordAuthentication no
PermitRootLogin prohibit-password
PubkeyAuthentication yes

systemctl restart sshd
```

**Verify before closing your terminal:** open a second terminal and confirm you can still SSH in.

## 3. Firewall (UFW)

UFW is the right abstraction for most VPS setups. Coolify needs ports 80 and 443. Lock down everything else.

```bash
ufw default deny incoming
ufw default allow outgoing

# SSH
ufw allow 22/tcp

# HTTP and HTTPS (Traefik handles these)
ufw allow 80/tcp
ufw allow 443/tcp

# Coolify dashboard (only while setting up, lock down after)
ufw allow 8000/tcp

ufw enable
ufw status
```

After Coolify is configured, remove port 8000 if you are accessing it via a Traefik route or Tailscale instead.

**Rule:** `ufw status` must show only ports that have active listeners. Audit after every deploy.

## 4. fail2ban

fail2ban bans IPs that repeatedly fail authentication. Default config covers SSH.

```bash
# /etc/fail2ban/jail.local
cat > /etc/fail2ban/jail.local << 'EOF'
[DEFAULT]
bantime  = 1h
findtime = 10m
maxretry = 5

[sshd]
enabled = true
EOF

systemctl enable fail2ban
systemctl start fail2ban
fail2ban-client status sshd
```

## 5. Docker

Install Docker Engine from the official Docker repo, not the Ubuntu snap.

```bash
curl -fsSL https://get.docker.com | sh

# Verify
docker --version
docker run --rm hello-world
```

No need for docker-compose as a standalone binary. Coolify uses the Docker CLI directly.

## 6. Coolify Installation

Coolify is a self-hosted PaaS. It manages containers, SSL, and deployments via a web UI.

```bash
curl -fsSL https://cdn.coollabs.io/coolify/install.sh | bash
```

This installs Coolify and starts it on port 8000. Access at `http://YOUR_SERVER_IP:8000`.

### Initial setup

1. Create your admin account
2. Add your server as a "localhost" server
3. Create projects (mad-house, orinadus, personal)
4. Connect your GitHub account for deployments

See `vps/coolify/coolify-deployment.ipynb` for full deployment workflow.

## 7. Unattended Upgrades

Keep the OS patched automatically for security updates.

```bash
dpkg-reconfigure --priority=low unattended-upgrades
# Select "Yes"
```

Edit `/etc/apt/apt.conf.d/50unattended-upgrades` to enable automatic reboots:

```
Unattended-Upgrade::Automatic-Reboot "true";
Unattended-Upgrade::Automatic-Reboot-Time "04:00";
```

## 8. File Permission Baseline

Every secret-containing file must be locked down before going live.

```bash
# .env files
chmod 600 /path/to/.env

# Secret directories
chmod 700 ~/.secrets/

# Systemd EnvironmentFiles
chmod 600 /etc/myservice.env
chown root:root /etc/myservice.env

# SSH private keys
chmod 600 ~/.ssh/id_ed25519
```

This baseline applies to every server and every service. No exceptions.

## 9. Domain and DNS

Point your domain to the server IP with an A record.

```
Type    Name    Value
A       @       YOUR_SERVER_IP
A       *       YOUR_SERVER_IP    (wildcard for subdomains)
```

DNS propagation takes a few minutes to 48 hours depending on TTL.

### Verify propagation

```bash
dig yourdomain.com +short
# Should return your server IP
```

Traefik (installed with Coolify) handles SSL certificates via Let's Encrypt automatically once DNS resolves.

## Setup Checklist

Run through this before declaring the server production-ready:

- [ ] SSH key auth working, password auth disabled
- [ ] UFW enabled, only necessary ports open
- [ ] fail2ban running and watching sshd
- [ ] Unattended upgrades configured
- [ ] Docker installed and working
- [ ] Coolify installed and accessible
- [ ] Domain DNS pointing to server
- [ ] Secret file permissions at 600/700
- [ ] `ufw status` shows no stale rules